## Introduction :

At the end of the data acquisition campaign, a large amount of raw data (physiological, stress-related and others) was collected. They were reunited and organized with [gen_0_raw_dataset_generation.py](../Code/gen_0_raw_dataset_generation.py) and other external processes to form the MIAMS_Raw_Dataset. It will be the starting point :

<img src="C:/Users/theghost/Desktop/Imvia_Internship/MIAMS/Code/analysis/readme_data/MIAMS_Raw_Dataset_Architecture.png" height="300"/>

This Jupyter Notebook will develop in detail the different steps to transform this Raw Dataset into a Pandas Dataframe usable to train artificial intelligence models. The goal is that this [Dataframe](../Dataframe/hbh_final_wl_3600_wss_3600_hrv_threshold_10pct_hrv_clean_data_False.parquet.gzip) contains a set of information with a corresponding stress score. Here is an overview :

<img src="C:/Users/theghost/Desktop/Imvia_Internship/MIAMS/Code/analysis/readme_data/Final_DataFrame.PNG" height="300"/>

## Imports :

In [1]:
import my_paths

from gen_1_physiological_features_computing import *
from gen_2_physiological_features_merging import *
from gen_3_meteorological_data_treatment import *
from gen_4_hbh_data_merging import *


KeyboardInterrupt



## Names and paths :

We first create a second dataset that will contain all the results and will be named MIAMS_features (MIAMS Computed dataset in figure below). 

In [ ]:
raw_dataset_path = 'C:/Users/theghost/Desktop/Imvia_Internship/MIAMS/MIAMS/MIAMS_Raw_Dataset'
computed_dataset_path = 'C:/Users/theghost/Desktop/Imvia_Internship/MIAMS/MIAMS/created'

## 1. Copy the original dataset architecture


This method recreate the architecture of a folder for the extracted featurs

First, to form this second dataset, we copy the architecture of the first one, that is to say all the directory tree but without the files. We obtain :

<img src="../readme_data/1_Computed_Dataset.PNG" height="300"/>

In [ ]:
my_paths.copy_folder_architecture(
    original_directory_path=raw_dataset_path,
    target_directory_path=computed_dataset_path
)

## 2. Physiological data descriptors calculation

First descriptors are calculated from each participant's sessions. The raw information is taken from MIAMS_raw and the results are stored as a Parquet file in MIAMS_features. This gives us one Parquet file per session :

<img src="../readme_data/2_Computed_Dataset.png" height="300"/>

The descriptors are calculated with the [Flirt library](https://flirt.readthedocs.io/en/latest/) and according to the [computing_parameters](../Code/computing_parameters.py) provided. These parameters can be found in the name given to the Parquets files. For the following, we decided to use the following parameters :

- window_length and window_step_size are set to one hour. 
- parameter hrv_threshold is set to 0.1 (low value) and hrv_clean_data is disabled because Empatica already filters the HRV signal. 
- Finally only_on_full_hour_slots is an option that ensures that descriptors are computed on hourly windows. For example from 8 am to 9 am or from 2 pm to 3 pm. This implies that not all the data is taken into account and therefore a loss of information (see [truncate_time_indexed_dataframe_to_have_only_full_hour_slots](./computing_parameters.py)). However, in this way, we obtain descriptors that are regular and comparable. Moreover, given that the participants only reported their stress twice a day, once at noon for the morning and once in the evening for the afternoon, it was not necessary for the physiological descriptors to be more precise.

In [ ]:
computing_parameters = Computing_Parameters(
        window_length=3600, 
        window_step_size=3600,
        hrv_threshold=0.1,
        hrv_clean_data=False,
        only_on_full_hour_slots=True
)
compute_features_for_all_dataset_sessions(
    raw_dataset_path=raw_dataset_path,
    destination_dataset_path=computed_dataset_path,
    computing_parameters=computing_parameters
)

In [ ]:
data=pd.read_csv("C:/Users/theghost/Desktop/Imvia_Internship/MIAMS/MIAMS/MIAMS_Raw_Dataset/sessions/user_16/session_1450893/BVP.csv")
data

In [ ]:
data.drop(data.index[:1], inplace=True)

In [ ]:
data.head()

In [ ]:
len(dd.iloc[:,0])

In [ ]:
print("number of recoreded hour for this session",len(dd.iloc[:,0])/3600)

In [ ]:
for window in dd.rolling(window= 3600,step=3600):
    print(window)
    print("--------------------------")

In [ ]:
from collections import deque
from itertools import islice

def sliding_window(iterable, size=2, step=1, fillvalue=None):
    if size < 0 or step < 1:
        raise ValueError
    it = iter(iterable)
    q = deque(islice(it, size), maxlen=size)
    if not q:
        return  # empty iterable or size == 0
    q.extend(fillvalue for _ in range(size - len(q)))  # pad to size
    while True:
        yield iter(q)  # iter() to avoid accidental outside modifications
        try:
            q.append(next(it))
        except StopIteration: # Python 3.5 pep 479 support
            return
        q.extend(next(it, fillvalue) for _ in range(step - 1))

In [ ]:
f=sliding_window(dd)
f

In [ ]:
#https://www.tensorflow.org/api_docs/python/tf/keras/utils/timeseries_dataset_from_arrayhttps://www.tensorflow.org/api_docs/python/tf/keras/utils/timeseries_dataset_from_array
import tensorflow as tf
d=tf.keras.utils.timeseries_dataset_from_array(
    dd,
    targets=None ,
    sequence_length=3600, #window size
    sequence_stride=3600, #window step (Period between successive output sequences) use this one
    #sampling_rate=, #window step (Period between individual timesteps within sequences)
    #batch_size=128,
    #shuffle=False,
    #seed=None,
    #start_index=None,
    #end_index=None
)
for batch in d:
    inputs = batch
    print(inputs)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots()
axs.set_title("Signal")
axs.plot(dd.iloc[:,0] , color='C0')
axs.set_xlabel("Time")
axs.set_ylabel("Amplitude")

plt.show()

## 3. Physiological data descriptors merge

Then we merge the results by participant : 

<img src="../readme_data/3_Computed_Dataset.png" height="300"/>

And finally we merge the results of all participants in one file :

<img src="../readme_data/4_Computed_Dataset.png" height="300"/>

Note : Errors may occur, so sessions may be skipped due to overlap. This happens when a session starts in time before the previous one ends. In this case, the overlapping session is ignored and the program continue. Although this is not really possible, it can happen because of a time-stamping error in Empatica wristbands. Indeed, the bracelets are not permanently connected, they set their internal clock to the right time at each synchronization with a computer. Between two synchronizations they can undergo a time-shifting.

In [ ]:
# merge by participant
merge_features_by_participant_for_all(
    dataset_path=computed_dataset_path,
    computing_parameters=computing_parameters
)

# merge all
merge_all_participants_features_in_a_dataframe(
    dataset_path=computed_dataset_path,
    computing_parameters=computing_parameters
)

## 4. Processing of meteorological data

The meteorological data of the 5 months during which the experimentation took place have been downloaded from this [website](https://donneespubliques.meteofrance.fr/). They are many features. This code get interesting features and stores them in a single file.
In addition, a rolling average is calculated for some of these indicators. It is the result of the difference between the average of the day and the average of the last 5 days. The purpose of this operation is to highlight changes in trends over time.

In [ ]:
group_meteorological_data_for_Dijon_from_raw_csv_files(
    raw_weather_csv_folder_path = my_paths.get_dataset_weather_directory_path(dataset_path=raw_dataset_path),
    destination_directory_path = my_paths.get_dataset_weather_directory_path(dataset_path=computed_dataset_path),
)

## 5. Final Merge
Here we merge the physiological features , Survey answers and Weather data.

Once all the different types of data have been processed (physiological, stress, meteorological), they are assembled to form the final dataframe.

We take advantage of this merging to add a last data type "date_infos", i.e. the date and the time. This indicates, for example, the day of the week, whether it is a working day or a rest day, or whether the participant was taking an exam at that time.

The principle is to start from the dataframe containing the physiological data of all participants. For each line of it, we append data related to stress, weather and others that correspond, then we move to the next line.

In [ ]:
merge_all(
    computed_dataset_path=computed_dataset_path,
    computing_parameters=computing_parameters
)
